In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [ ]:
# Preset file

filename = "CRMLS_0525-0526_clean.csv"

In [ ]:
def load_df(file=filename):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  df["CloseDate"] = pd.to_datetime(df["CloseDate"])     # Converts "CloseDate" values to datetime type
  return df

In [ ]:
# Preset Values

main_cols = ["BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet",
             "DaysOnMarket", "YearBuilt", "PostalCode", "SaleMonth"]

extras = ["ViewYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN"]
extra_cols = [a+"_True" for a in extras] + [a+"_False" for a in extras]

totals = main_cols+extra_cols

target = "ClosePrice"

In [ ]:
def test_train_split(df):
  """
  Takes a DataFrame
  Encodes "PropertyType" column
  Returns a defined training and test set for the DataFrame
  """
  yr_mo = []
  for i in df['CloseDate']:                                         # For each date in the "CloseDate" column
    yr, mo = i.year, i.month                                          # Define the year and month values of date i
    yr_mo.append([yr,mo])                                             # Append to "yr_mo" a list of date i's year and month
  te_set = [b for b in range(len(yr_mo)) if yr_mo[b] == [2026,5]]   # Define a list of row #s with date 05/2026
  te_rng = te_set[0::len(te_set)-1]                                 # Define a list of the first and last row in "te_set"
  tr, te = df[0:te_rng[0]], df[te_rng[0]:te_rng[1]]             # Define the training and test sets of the inputted df

  return tr, te

In [ ]:
def TreeReg(tr, te, feat=main_cols, targ=target):
  """
  Takes a training DataFrame and test DataFrame
  """
  x_tr = tr[feat].values            # Define the x_train set values
  y_tr = tr[targ].values            # Define the y_train set values
  x_te = te[feat].values            # Define the x_test set values
  y_te = te[targ].values            # Define the y_test set values

  model = DecisionTreeRegressor(random_state=0)        # Define Decision Tree Regression model
  model.fit(x_tr, y_tr)             # Fit training data to model
  y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

  r2 = r2_score(y_te, y_pred)       # Computes the r2 score of y_test and the predicted y

  return r2

In [ ]:
def ForestReg(tr, te, feat=main_cols, targ=target):
  """
  Takes a training DataFrame and test DataFrame
  """
  x_tr = tr[feat].values            # Define the x_train set values
  y_tr = tr[targ].values            # Define the y_train set values
  x_te = te[feat].values            # Define the x_test set values
  y_te = te[targ].values            # Define the y_test set values

  model = RandomForestRegressor(random_state=0)        # Define Random Forest Regression model
  model.fit(x_tr, y_tr)             # Fit training data to model
  y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

  r2 = r2_score(y_te, y_pred)       # Computes the r2 score of y_test and the predicted y

  return r2

# **Decision Tree Regressor**

***Non-Transform DataFrame***

In [ ]:
def main():
  main_df = load_df()
  train, test = test_train_split(main_df)

  r2_scores = []
  for col in totals:
    r2s = TreeReg(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = TreeReg(train, test, cols)
    print(f"Correlation of {cols}: \t\t {round(r2s,4)}")

In [ ]:
if __name__ == "__main__":
  main()

Correlation of ['PostalCode']: 		 0.2616
Correlation of ['PostalCode', 'BathroomsTotalInteger']: 		 0.267
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 -2.5558
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea']: 		 -40.1345
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False']: 		 -14.8214
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 -14.8033
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 -14.8124
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True']: 		 -14.7513
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivate

In [ ]:
main_df = load_df()
train, test = test_train_split(main_df)

try_cols = ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True',
            'PoolPrivateYN_True', 'ViewYN_True', 'ViewYN_False', 'YearBuilt', 'NewConstructionYN_True', 'NewConstructionYN_False']

r2s = TreeReg(train, test, try_cols)
print(f"{round(r2s,4)}")

0.3905


***Results***

Highest R2 Score:  **0.3905**

* Included Features:

  - ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'ViewYN_True', 'ViewYN_False', 'YearBuilt','NewConstructionYN_True', 'NewConstructionYN_False']

* Excluded Features:

  - ['SaleMonth', 'DaysOnMarket']

---

* Wide range of R2 scores for different combination of features


***Log Transform DataFrame***

In [ ]:
def log_transform(df, feat=main_cols, targ=target):
  for col in feat+[targ]:
    new_col = []
    for val in df[col]:
      new_col.append(float(np.log(val)))
    df[col] = pd.DataFrame(new_col)
  return df

In [ ]:
def main():
  main_df = load_df()
  log_df = main_df.copy()
  log_df = log_transform(log_df)
  train, test = test_train_split(log_df)

  r2_scores = []
  for col in totals:
    r2s = TreeReg(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = TreeReg(train, test, cols)
    print(f"Correlation of {cols}: \t\t {round(r2s,4)}")

In [ ]:
if __name__ == "__main__":
  main()

Correlation of ['PostalCode']: 		 0.7341
Correlation of ['PostalCode', 'LivingArea']: 		 0.797
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger']: 		 0.7943
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 0.7974
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False']: 		 0.7892
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.7883
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False']: 		 0.7869
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'YearBuilt']: 		 0.7742
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPriva

In [ ]:
main_df = load_df()
log_df = main_df.copy()
log_df = log_transform(log_df)
train, test = test_train_split(log_df)

try_cols = ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'NewConstructionYN_False']

r2s = TreeReg(train, test, try_cols)
print(f"{round(r2s,4)}")

0.7995


***Results***

Highest R2 Score:  **0.7995**

* Included Features:

  - ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'NewConstructionYN_False']

* Excluded Features:

  - ['PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'YearBuilt', 'SaleMonth', 'DaysOnMarket']

---

* All R2 scores are within a reasonable range, and do not deviate significantly from one another


# **Random Forest Regressor**

***Non-Transform DataFrame***

In [ ]:
def main():
  main_df = load_df()
  train, test = test_train_split(main_df)

  r2_scores = []
  for col in totals:
    r2s = ForestReg(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = ForestReg(train, test, cols)
    print(f"Correlation of {cols}: \t\t {round(r2s,4)}")

In [ ]:
if __name__ == "__main__":
  main()

Correlation of ['BathroomsTotalInteger']: 		 0.2601
Correlation of ['BathroomsTotalInteger', 'PostalCode']: 		 0.3277
Correlation of ['BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal']: 		 -2.8966
Correlation of ['BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False']: 		 -2.9102
Correlation of ['BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 -2.813
Correlation of ['BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 -2.8151
Correlation of ['BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True']: 		 -2.8232
Correlation of ['BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'ViewYN_True']: 		 -5.6107
Correlation of ['BathroomsTotalInteger', 'PostalCode', 'Bedro

In [ ]:
main_df = load_df()
train, test = test_train_split(main_df)

try_cols = ['BathroomsTotalInteger', 'PostalCode', 'YearBuilt', 'SaleMonth', 'NewConstructionYN_False']

r2s = ForestReg(train, test, try_cols)
print(f"{round(r2s,4)}")

0.3911


***Results***

Highest R2 Score:  **0.3911**

* Included Features:

  - ['BathroomsTotalInteger', 'PostalCode', 'YearBuilt', 'SaleMonth', 'NewConstructionYN_False']

* Excluded Features:

  - ['BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'ViewYN_True', 'ViewYN_False', 'NewConstructionYN_True', 'DaysOnMarket']

---

* Less wide-ranging values of R2 scores than Decision Tree Regressor


***Log Transform DataFrame***

In [ ]:
def main():
  main_df = load_df()
  log_df = main_df.copy()
  log_df = log_transform(log_df)
  train, test = test_train_split(log_df)

  r2_scores = []
  for col in totals:
    r2s = ForestReg(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = ForestReg(train, test, cols)
    print(f"Correlation of {cols}: \t\t {round(r2s,4)}")

In [ ]:
if __name__ == "__main__":
  main()

Correlation of ['PostalCode']: 		 0.7341
Correlation of ['PostalCode', 'LivingArea']: 		 0.8669
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger']: 		 0.8686
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 0.8714
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False']: 		 0.8714
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.8715
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False']: 		 0.8729
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LotSizeSquareFeet']: 		 0.8823
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', '

In [ ]:
main_df = load_df()
log_df = main_df.copy()
log_df = log_transform(log_df)
train, test = test_train_split(log_df)

try_cols = ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_True',
            'PoolPrivateYN_False', 'LotSizeSquareFeet', 'YearBuilt', 'PoolPrivateYN_True', 'NewConstructionYN_True']

r2s = ForestReg(train, test, try_cols)
print(f"{round(r2s,4)}")

0.8842


***Results***

Highest R2 Score:  **0.8842**

* Included Features:

  - ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LotSizeSquareFeet', 'YearBuilt', 'PoolPrivateYN_True']

  - ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LotSizeSquareFeet', 'YearBuilt', 'PoolPrivateYN_True', 'NewConstructionYN_True']

* Excluded Features:

  - ['ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'NewConstructionYN_False', 'SaleMonth', 'DaysOnMarket']

  - ['ViewYN_False', FireplaceYN_False, 'ViewYN_True', 'NewConstructionYN_False', 'SaleMonth', 'DaysOnMarket']

---

* All R2 scores are within a reasonable range, and do not deviate significantly from one another
* Less wide-ranging values of R2 scores than Decision Tree Regressor


# **Week 6 - Feature Engineering**

***Features:***

**Ratios:**

* "ClosePrice" to "DaysOnMarket"
* "LivingArea" to "LotSizeSquareFeet"
  - "LotSizeSquareFeet" to "DaysOnMarket"
  - "LivingArea" to "DaysOnMarket"
* "BathroomsTotalInteger" to "BedroomsTotal"
  - "BathroomsTotalInteger" to "ClosePrice"
  - "BedroomsTotal" to "ClosePrice"

---
**Other:**

* Average "ClosePrice" per "PostalCode" (.groupby())
* Age of Propery (Current Year - "YearBuilt")


In [ ]:
combos = [["ClosePrice", "DaysOnMarket"], ["LivingArea", "LotSizeSquareFeet"], ["LotSizeSquareFeet", "DaysOnMarket"], ["LivingArea", "DaysOnMarket"],
          ["BathroomsTotalInteger", "BedroomsTotal"], ["BathroomsTotalInteger", "ClosePrice"], ["BedroomsTotal", "ClosePrice"]]

new_names = [["ClosePrice/DaysOnMarket"], ["LivingArea/LotSizeSquareFeet"], ["LotSizeSquareFeet/DaysOnMarket"], ["LivingArea/DaysOnMarket"],
             ["BathroomsTotalInteger/BedroomsTotal"], ["BathroomsTotalInteger/ClosePrice"], ["BedroomsTotal/ClosePrice"]]

CRMLS_df = main_df.copy()
a=0
for set in combos:
  feat = [main_df[set[0]][i]/main_df[set[1]][i] for i in range(len(main_df))]
  CRMLS_df.insert(CRMLS_df.shape[1], new_names[a][0], feat)
  a += 1

# **Geographic Layer of School Districts**

* Imported GeoJSON file
---
* "Latitude" != 0
  - Also eliminated "Longitude" values equal to 0
* "DistrictType" == "Unified" (reduced rows from 937 to 345)
---
* Converted cleaned CRMLS (main_df) "Longitude", "Latitude" to Geometric Array of geographic points
* Created copy of main_df with new column "geometry" for geographic points
* Converted copy of main_df into a GeoDataFrame (geo_gdf) with "geometry" as geometric column
---
* Defined and spatially joined geo_gdf and school district gdf (schl_gdf) as combo_gdf
* Redefined combo_gdf with all main_df columns and only "DistrictName" column from schl_gdf

In [ ]:
import geopandas as gpd

In [ ]:
# Preset file

geo_file = "California_School_District_Areas_2024-25.geojson"

enriched_df = "CRMLS_0525-0526_enriched.csv"

In [ ]:
def load_data(file=geo_file):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  gdf = gpd.read_file(file)
  return gdf

In [ ]:
CRMLS_df = load_df()
schl_gdf = load_data()

main_df = CRMLS_df[CRMLS_df['Latitude'] != 0]
schl_gdf = schl_gdf[schl_gdf["DistrictType"] == "Unified"]
schl_gdf.reset_index(drop=True)

geo = gpd.points_from_xy(x=main_df.Longitude, y=main_df.Latitude, crs=schl_gdf.crs)
geo_main = main_df.copy()
geo_main["geometry"] = geo
geo_gdf = gpd.GeoDataFrame(geo_main).set_geometry(col="geometry")

combo_gdf = gpd.sjoin(geo_gdf, schl_gdf)
combo_gdf = combo_gdf[main_df.columns.to_list()+["DistrictName"]]
combo_gdf

,Flooring,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,...,ViewYN_False,ViewYN_True,PoolPrivateYN_False,PoolPrivateYN_True,FireplaceYN_False,FireplaceYN_True,NewConstructionYN_False,NewConstructionYN_True,SaleMonth,DistrictName
1,"Carpet,Stone,Wood",2288000.0,1114680429,2025-05-05,2175000.0,34.129941,-118.426100,13438 Java Drive,Residential,2247.0,...,0,1,0,1,0,1,1,0,5,Los Angeles Unified
2,NaN,5975000.0,1114670553,2025-05-29,5975000.0,34.135406,-118.118478,1538 E California Boulevard,Residential,3993.0,...,1,0,0,0,0,1,1,0,5,Pasadena Unified
3,"Carpet,Tile,Wood",1700000.0,1114650212,2025-05-28,1900000.0,34.037751,-118.395258,2807 Cardiff Avenue,Residential,1829.0,...,1,0,1,0,0,1,1,0,5,Los Angeles Unified
4,"Carpet,Laminate",640000.0,1114573617,2025-05-27,640000.0,34.247108,-118.428730,13535 Goleta Street,Residential,1440.0,...,1,0,1,0,1,0,1,0,5,Los Angeles Unified
5,Wood,1000000.0,1114441822,2025-05-22,1000000.0,33.981880,-118.356613,4629 W 64th Street,Residential,1228.0,...,0,1,1,0,1,0,1,0,5,Los Angeles Unified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133203,NaN,800000.0,1065799115,2026-05-07,455000.0,33.786867,-118.187745,1584 Elm Avenue,Residential,1724.0,...,1,0,1,0,1,0,1,0,5,Long Beach Unified
133204,Wood,3500000.0,1063517703,2026-05-08,1850000.0,34.377680,-119.145860,6770 Wheeler Canyon Road,Residential,3000.0,...,0,1,1,0,0,1,1,0,5,Santa Paula Unified
133205,Wood,675000.0,1052367803,2026-05-06,655000.0,34.275262,-118.452369,14655 Maclay Street,Residential,1008.0,...,0,1,1,0,0,1,1,0,5,Los Angeles Unified
133206,"Carpet,SeeRemarks,Stone,Wood",35000000.0,1048618285,2026-05-15,21500000.0,37.854648,-121.961587,7 Country Oak Ln,Residential,23314.0,...,0,0,1,0,0,1,1,0,5,San Ramon Valley Unified


In [ ]:
def save_csv(df, file=enriched_df):
  """
  Takes a DataFrame
  Saves the inputted DataFrame as a .csv file, given inputted name
  """
  df.to_csv(filename, index=False)

In [ ]:
save_csv(combo_gdf)

# ***Model Re-evalutation***

In [ ]:
print("Changes in Data Quantity")
print("------------------------ \n")

print(f"Starting length of cleaned CRMLS DataFrame: \t {len(CRMLS_df)} rows \n")

print(f"Eliminating zero-valued 'Longitude', 'Latitude': \t {len(CRMLS_df) - len(main_df)} rows dropped \n")

print(f"Length of spatially joined DataFrame: \t {len(combo_gdf)} rows")
print(f"Spatial join dropped an additional \t {len(main_df) - len(combo_gdf)} rows")

Changes in Data Quantity
------------------------ 

Starting length of cleaned CRMLS DataFrame: 	 133208 rows 

Eliminating zero-valued 'Longitude', 'Latitude': 	 16 rows dropped 

Length of spatially joined DataFrame: 	 101357 rows
Spatial join dropped an additional 	 31835 rows


In [ ]:
CRMLS_df = pd.get_dummies(data=CRMLS_df, columns=["DistrictName"])

In [ ]:
new_names = [["ClosePrice/DaysOnMarket"], ["LivingArea/LotSizeSquareFeet"], ["LotSizeSquareFeet/DaysOnMarket"], ["LivingArea/DaysOnMarket"],
             ["BathroomsTotalInteger/BedroomsTotal"], ["BathroomsTotalInteger/ClosePrice"], ["BedroomsTotal/ClosePrice"]]

geo_cols = [i for i in CRMLS_df.columns if "DistrictName" in i]

all_cols = totals+[i[0] for i in new_names]+geo_cols

# **Linear Regression**

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
def LinReg(tr, te, feat=totals, targ=target):
  """
  Takes a training DataFrame, test DataFrame, and feature and target values
  Returns R2 score of the Linear Regression
  """
  x_tr = tr[feat].values            # Define the x_train set values
  y_tr = tr[targ].values            # Define the y_train set values
  x_te = te[feat].values            # Define the x_test set values
  y_te = te[targ].values            # Define the y_test set values

  model = LinearRegression()        # Define Linear Regression model
  model.fit(x_tr, y_tr)             # Fit training data to model
  y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

  r2 = r2_score(y_te, y_pred)       # Computes the r2 score of y_test and the predicted y

  return r2

In [ ]:
def main():
  train, test = test_train_split(CRMLS_df)

  r2_scores = []
  for col in all_cols:
    r2s = LinReg(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[all_cols[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  r2_totals = []
  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = LinReg(train, test, cols)
    r2_totals.append([cols, r2s])
  r2_totals = sorted(r2_totals, key=lambda x: x[1], reverse=True)
  print(f"Correlation of {r2_totals[0][1]}: \t\t {round(r2_totals[0][0],4)}")

In [ ]:
if __name__ == "__main__":
  main()

TypeError: type list doesn't define __round__ method

**Log Transform of Linear Regression**

In [ ]:
def main():
  log_df = CRMLS_df.copy()
  log_df = log_transform(log_df)
  train, test = test_train_split(log_df)

  r2_scores = []
  for col in totals:
    r2s = LinReg(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  r2_totals = []
  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = LinReg(train, test, cols)
    r2_totals.append([cols, r2s])
  r2_totals = sorted(r2_totals, key=lambda x: x[1], reverse=True)
  print(f"Correlation of {r2_totals[0][1]}: \t\t {round(r2_totals[0][0],4)}")

In [ ]:
if __name__ == "__main__":
  main()

Correlation of ['LivingArea']: 		 0.3755
Correlation of ['LivingArea', 'BathroomsTotalInteger']: 		 0.3791
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 0.3796
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False']: 		 0.3893
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.3894
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False']: 		 0.3955
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'PoolPrivateYN_True']: 		 0.416
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'PoolPrivateYN_True', 'PostalCode']: 		 0.4481
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN

**Results**

***Non-Transform***

*Original* Highest R2 Score:  **0.3322**

*Enriched* Highest R2 Score:  **0.3461**

* Increased R2 Score

---
***Log Transform***

*Original* Highest R2 Score:  **0.5602**

*Enriched* Highest R2 Score:  **0.5573**

* Decreased R2 Score

# **Decision Tree Regression**

In [ ]:
def main():
  train, test = test_train_split(CRMLS_df)

  r2_scores = []
  for col in totals:
    r2s = TreeReg(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  r2_totals = []
  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = TreeReg(train, test, cols)
    r2_totals.append([cols, r2s])
  r2_totals = sorted(r2_totals, key=lambda x: x[1], reverse=True)
  print(f"Correlation of {r2_totals[0][1]}: \t\t {round(r2_totals[0][0],4)}")

In [ ]:
if __name__ == "__main__":
  main()

Correlation of ['PostalCode']: 		 0.2616
Correlation of ['PostalCode', 'BathroomsTotalInteger']: 		 0.267
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 -2.5558
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea']: 		 -40.1345
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False']: 		 -14.8214
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 -14.8033
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 -14.8124
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True']: 		 -14.7513
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivate

In [ ]:
train, test = test_train_split(CRMLS_df)

try_cols = ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True',
            'PoolPrivateYN_True', 'ViewYN_True', 'ViewYN_False', 'YearBuilt', 'NewConstructionYN_True', 'NewConstructionYN_False']

r2s = TreeReg(train, test, try_cols)
print(f"{round(r2s,4)}")

0.3905


***Log Transform DataFrame***

In [ ]:
def main():
  log_df = CRMLS_df.copy()
  log_df = log_transform(log_df)
  train, test = test_train_split(log_df)

  r2_scores = []
  for col in totals:
    r2s = TreeReg(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  r2_totals = []
  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = TreeReg(train, test, cols)
    r2_totals.append([cols, r2s])
  r2_totals = sorted(r2_totals, key=lambda x: x[1], reverse=True)
  print(f"Correlation of {r2_totals[0][1]}: \t\t {round(r2_totals[0][0],4)}")

In [ ]:
if __name__ == "__main__":
  main()

Correlation of ['PostalCode']: 		 0.7341
Correlation of ['PostalCode', 'LivingArea']: 		 0.797
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger']: 		 0.7943
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 0.7974
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False']: 		 0.7892
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.7883
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False']: 		 0.7869
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'YearBuilt']: 		 0.7742
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPriva

In [ ]:
log_df = CRMLS_df.copy()
log_df = log_transform(log_df)
train, test = test_train_split(log_df)

try_cols = ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'NewConstructionYN_False']

r2s = TreeReg(train, test, try_cols)
print(f"{round(r2s,4)}")

0.7995


**Results**


***Non-Transform***

*Original* Highest R2 Score:  **0.3905**

*Enriched* Highest R2 Score:  **0.3905**

 * Unchanged R2 Score

---
***Log Transforms***

*Original* Highest R2 Score:  **0.7995**

*Enriched* Highest R2 Score:  

**Random Forest Regression**